# Consigna del desafío 4

Replicar y extender el traductor:
- Replicar el modelo en PyTorch.
- Extender el entrenamiento a más datos y tamaños de 
secuencias mayores.
- Explorar el impacto de la cantidad de neuronas en 
las capas recurrentes.
- Mostrar 5 ejemplos de traducciones generadas.
- Extras que se pueden probar: Embeddings 
pre-entrenados para los dos idiomas; cambiar la 
estrategia de generación (por ejemplo muestreo 
aleatorio); 

---
# Imports
---

In [ ]:
import numpy as np
import pandas as pd
import re

import torch
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F


from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

c:\Users\julia\anaconda3\envs\PLN\Lib\site-packages\torchtext\__init__.py:7: SyntaxWarning: invalid escape sequence '\ '
  "\n/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ \n"


OSError: [WinError 127] No se encontró el proceso especificado

---
# Dataset
--- 

In [ ]:
text_file = "desafios/spa-eng/spa.txt"

with open(text_file) as f:
    lines = f.read().split("\n")[:-1]

MAX_NUM_SENTENCES = 6000

np.random.seed([40])
np.random.shuffle(lines)

input_sentences = []
output_sentences = []
output_sentences_inputs = []
count = 0

for line in lines:
    count += 1
    if count > MAX_NUM_SENTENCES:
        break

    if '\t' not in line:
        continue

    input_sentence, output = line.rstrip().split('\t')

    output_sentence = output + ' <eos>'
    output_sentences_input = '<sos> ' + output

    input_sentences.append(input_sentence)
    output_sentences.append(output_sentence)
    output_sentences_inputs.append(output_sentences_input)

print("Cantidad de rows disponibles:", len(lines))
print("Cantidad de rows utilizadas:", len(input_sentences))

In [ ]:
input_sentences[0], output_sentences[0], output_sentences_inputs[0]

---
# Preprocesamiento
---

## Tokenizador ingles

In [ ]:
MAX_VOCAB_SIZE = 8000

In [ ]:
input_tokenizer = get_tokenizer('basic_english')

In [ ]:
def yield_tokens(data_iter, tokenizer):
    for text in data_iter:
        yield tokenizer(text)

In [ ]:
vocab = build_vocab_from_iterator(
    yield_tokens(input_sentences),
    max_tokens=MAX_VOCAB_SIZE,         
    specials=['<unk>']                 
)

In [ ]:
vocab.set_default_index(vocab['<unk>'])

In [ ]:
input_integer_seq = [vocab(input_tokenizer(text)) for text in input_sentences]

In [ ]:
word2idx_inputs = vocab.get_stoi()
print(f"Palabras en el vocabulario: {len(word2idx_inputs)}")
print("Diccionario (word_index):", word2idx_inputs)


In [ ]:
max_input_len = max(len(sen) for sen in input_integer_seq)
print("\nSentencia de entrada más larga:", max_input_len)

print("\nSecuencias de enteros generadas:")
print(input_integer_seq)

## Tokenizador español

In [ ]:
def custom_filter_tokenizer(text):
    clean_text = re.sub(r'[^a-zA-Z0-9<> ]', '', text.lower())
    
    # Dividir el texto limpio en tokens por espacio
    return clean_text.split()

In [ ]:
vocab = build_vocab_from_iterator(
    yield_tokens(output_sentences),
    max_tokens=MAX_VOCAB_SIZE,
    specials=['<unk>', '<sos>', '<eos>']
)
vocab.set_default_index(vocab['<unk>'])

In [ ]:
output_integer_seq = [vocab(custom_filter_tokenizer(s)) for s in output_sentences]
output_input_integer_seq = [vocab(custom_filter_tokenizer(s)) for s in output_sentences_inputs]

In [ ]:
word2idx_outputs = vocab.get_stoi()

In [ ]:
print("Palabras en el vocabulario:", len(word2idx_outputs))
print("Diccionario (word_index):", word2idx_outputs)

num_words_output = len(vocab)
print("Tamaño final del vocabulario:", num_words_output)

max_out_len = max(len(sen) for sen in output_integer_seq)
print("Sentencia de salida más larga:", max_out_len)

print("\nSecuencias de enteros (output):", output_integer_seq)
print("Secuencias de enteros (input):", output_input_integer_seq)

In [ ]:
max_input_len = 16
max_out_len = 18

In [ ]:
def pad_sequences_pre(sequences, maxlen):
    padded = np.zeros((len(sequences), maxlen), dtype=np.int64)
    for i, seq in enumerate(sequences):
        padded[i, -len(seq):] = seq
    return torch.from_numpy(padded)


In [ ]:
encoder_input_sequences = pad_sequences_pre(input_integer_seq, maxlen=max_input_len)
sequences_as_tensors = [torch.tensor(s) for s in output_input_integer_seq]
decoder_input_sequences = pad_sequence(sequences_as_tensors, batch_first=True, padding_value=0)

In [ ]:
sequences_as_tensors = [torch.tensor(s) for s in output_integer_seq]
decoder_output_sequences_long = pad_sequence(sequences_as_tensors)
decoder_targets = F.one_hot(decoder_output_sequences_long, num_classes=num_words_output)

---
# Preparacion de embeddings
---

In [ ]:
import os
import gdown
if os.access('gloveembedding.pkl', os.F_OK) is False:
    url = 'https://drive.google.com/uc?id=1KY6avD5I1eI2dxQzMkR3WExwKwRq2g94&export=download'
    output = 'gloveembedding.pkl'
    gdown.download(url, output, quiet=False)
else:
    print("Los embeddings gloveembedding.pkl ya están descargados")

In [ ]:
import logging
import os
from pathlib import Path
from io import StringIO
import pickle

class WordsEmbeddings(object):
    logger = logging.getLogger(__name__)

    def __init__(self):
        # load the embeddings
        words_embedding_pkl = Path(self.PKL_PATH)
        if not words_embedding_pkl.is_file():
            words_embedding_txt = Path(self.WORD_TO_VEC_MODEL_TXT_PATH)
            assert words_embedding_txt.is_file(), 'Words embedding not available'
            embeddings = self.convert_model_to_pickle()
        else:
            embeddings = self.load_model_from_pickle()
        self.embeddings = embeddings
        # build the vocabulary hashmap
        index = np.arange(self.embeddings.shape[0])
        # Dicctionarios para traducir de embedding a IDX de la palabra
        self.word2idx = dict(zip(self.embeddings['word'], index))
        self.idx2word = dict(zip(index, self.embeddings['word']))

    def get_words_embeddings(self, words):
        words_idxs = self.words2idxs(words)
        return self.embeddings[words_idxs]['embedding']

    def words2idxs(self, words):
        return np.array([self.word2idx.get(word, -1) for word in words])

    def idxs2words(self, idxs):
        return np.array([self.idx2word.get(idx, '-1') for idx in idxs])

    def load_model_from_pickle(self):
        self.logger.debug(
            'loading words embeddings from pickle {}'.format(
                self.PKL_PATH
            )
        )
        max_bytes = 2**28 - 1 # 256MB
        bytes_in = bytearray(0)
        input_size = os.path.getsize(self.PKL_PATH)
        with open(self.PKL_PATH, 'rb') as f_in:
            for _ in range(0, input_size, max_bytes):
                bytes_in += f_in.read(max_bytes)
        embeddings = pickle.loads(bytes_in)
        self.logger.debug('words embeddings loaded')
        return embeddings

    def convert_model_to_pickle(self):
        # create a numpy strctured array:
        # word     embedding
        # U50      np.float32[]
        # word_1   a, b, c
        # word_2   d, e, f
        # ...
        # word_n   g, h, i
        self.logger.debug(
            'converting and loading words embeddings from text file {}'.format(
                self.WORD_TO_VEC_MODEL_TXT_PATH
            )
        )
        structure = [('word', np.dtype('U' + str(self.WORD_MAX_SIZE))),
                     ('embedding', np.float32, (self.N_FEATURES,))]
        structure = np.dtype(structure)
        # load numpy array from disk using a generator
        with open(self.WORD_TO_VEC_MODEL_TXT_PATH, encoding="utf8") as words_embeddings_txt:
            embeddings_gen = (
                (line.split()[0], line.split()[1:]) for line in words_embeddings_txt
                if len(line.split()[1:]) == self.N_FEATURES
            )
            embeddings = np.fromiter(embeddings_gen, structure)
        # add a null embedding
        null_embedding = np.array(
            [('null_embedding', np.zeros((self.N_FEATURES,), dtype=np.float32))],
            dtype=structure
        )
        embeddings = np.concatenate([embeddings, null_embedding])
        # dump numpy array to disk using pickle
        max_bytes = 2**28 - 1 # # 256MB
        bytes_out = pickle.dumps(embeddings, protocol=pickle.HIGHEST_PROTOCOL)
        with open(self.PKL_PATH, 'wb') as f_out:
            for idx in range(0, len(bytes_out), max_bytes):
                f_out.write(bytes_out[idx:idx+max_bytes])
        self.logger.debug('words embeddings loaded')
        return embeddings


class GloveEmbeddings(WordsEmbeddings):
    WORD_TO_VEC_MODEL_TXT_PATH = 'glove.twitter.27B.50d.txt'
    PKL_PATH = 'gloveembedding.pkl'
    N_FEATURES = 50
    WORD_MAX_SIZE = 60

class FasttextEmbeddings(WordsEmbeddings):
    WORD_TO_VEC_MODEL_TXT_PATH = 'cc.en.300.vec'
    PKL_PATH = 'fasttext.pkl'
    N_FEATURES = 300
    WORD_MAX_SIZE = 60

In [ ]:
# Por una cuestion de RAM se utilizarán los embeddings de Glove de dimension 50
model_embeddings = GloveEmbeddings()

In [ ]:
# Crear la Embedding matrix de las secuencias
# en inglés

print('preparing embedding matrix...')
embed_dim = model_embeddings.N_FEATURES
words_not_found = []

# word_index provieen del tokenizer

nb_words = min(MAX_VOCAB_SIZE, len(word2idx_inputs)) # vocab_size
embedding_matrix = np.zeros((nb_words, embed_dim))
for word, i in word2idx_inputs.items():
    if i >= nb_words:
        continue
    embedding_vector = model_embeddings.get_words_embeddings(word)[0]
    if (embedding_vector is not None) and len(embedding_vector) > 0:
        
        embedding_matrix[i] = embedding_vector
    else:
        # words not found in embedding index will be all-zeros.
        words_not_found.append(word)

print('number of null word embeddings:', np.sum(np.sum(embedding_matrix**2, axis=1) == 0))

In [ ]:
# Dimensión de los embeddings de la secuencia en inglés
embedding_matrix.shape